# YOLO-NAS Real-Time Detection (Colab GPU)
Detects **Tips** and **Liquid** with liquid percentage per tip.

### Steps:
1. Install dependencies
2. Download model from Google Drive
3. Upload your video
4. Run detection on GPU
5. Download / play the output video

## 1. Install Dependencies

In [ ]:
!pip install -q Cython numpy
!pip install -q pycocotools
!pip install -q super-gradients opencv-python-headless

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE'}")
print(f"CUDA available: {torch.cuda.is_available()}")


## 2. Download Model from Google Drive
This downloads `ckpt_best.pth` (~850MB) directly to the Colab runtime.

In [ ]:
import gdown

MODEL_URL = "https://drive.google.com/uc?id=1XPc4e1gyOBGoK1aCCS8Wu2kcM8kH-QLr"
MODEL_PATH = "ckpt_best.pth"

gdown.download(MODEL_URL, MODEL_PATH, quiet=False)
print(f"Model downloaded: {MODEL_PATH}")

## 3. Upload Your Video
Run this cell and select your video file.

In [ ]:
from google.colab import files

uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {VIDEO_PATH}")

**Or** mount Google Drive and set the path manually:
```python
from google.colab import drive
drive.mount('/content/drive')
VIDEO_PATH = '/content/drive/MyDrive/path/to/your/video.mp4'
```

## 4. Load Model on GPU

In [ ]:
from super_gradients.training import models

CLASSES = ["Tip", "Liquid"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model on {DEVICE}...")
model = models.get(
    "yolo_nas_l",
    num_classes=len(CLASSES),
    checkpoint_path=MODEL_PATH,
)
model = model.to(DEVICE)
model.eval()
print(f"Model loaded on {DEVICE}!")

## 5. Run Detection & Save Output Video

In [ ]:
import cv2
import time
import numpy as np

COLORS = {"Tip": (0, 255, 0), "Liquid": (255, 0, 0)}
CONF_THRESHOLD = 0.4
IOU_THRESHOLD = 0.5
OUTPUT_PATH = "output_detection.mp4"


def compute_iou(box_a, box_b):
    xa = max(box_a[0], box_b[0])
    ya = max(box_a[1], box_b[1])
    xb = min(box_a[2], box_b[2])
    yb = min(box_a[3], box_b[3])
    inter = max(0, xb - xa) * max(0, yb - ya)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0


def match_liquid_to_tips(tip_boxes, liquid_boxes):
    matches = {}
    for liq_box in liquid_boxes:
        best_idx, best_iou = -1, 0
        for i, tip_box in enumerate(tip_boxes):
            iou = compute_iou(tip_box, liq_box)
            if iou > best_iou:
                best_iou = iou
                best_idx = i
        if best_idx >= 0 and best_iou > 0.05:
            matches[best_idx] = liq_box
    return matches


def draw_detections(frame, pred, conf_threshold):
    bboxes = pred.prediction.bboxes_xyxy
    confidences = pred.prediction.confidence
    labels = pred.prediction.labels

    tip_boxes, liquid_boxes = [], []
    tip_count, liquid_count = 0, 0

    for bbox, conf, label in zip(bboxes, confidences, labels):
        if conf < conf_threshold:
            continue
        coords = [float(bbox[0]), float(bbox[1]), float(bbox[2]), float(bbox[3])]
        class_name = CLASSES[int(label)]
        if class_name == "Tip":
            tip_boxes.append(coords)
            tip_count += 1
        else:
            liquid_boxes.append(coords)
            liquid_count += 1

    tip_boxes.sort(key=lambda b: (b[0] + b[2]) / 2)
    liquid_matches = match_liquid_to_tips(tip_boxes, liquid_boxes)

    for i, tb in enumerate(tip_boxes):
        x1, y1, x2, y2 = map(int, tb)
        cv2.rectangle(frame, (x1, y1), (x2, y2), COLORS["Tip"], 2)
        tip_height = tb[3] - tb[1]

        if i in liquid_matches:
            lb = liquid_matches[i]
            liq_height = lb[3] - lb[1]
            pct = min((liq_height / tip_height * 100) if tip_height > 0 else 0, 100.0)
            lx1, ly1, lx2, ly2 = map(int, lb)
            cv2.rectangle(frame, (lx1, ly1), (lx2, ly2), COLORS["Liquid"], 2)
            text = f"Tip {i+1}: {pct:.0f}%"
            color = COLORS["Liquid"]
        else:
            text = f"Tip {i+1}: Empty"
            pct = 0
            color = COLORS["Tip"]

        ts = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)[0]
        cv2.rectangle(frame, (x1, y1 - ts[1] - 8), (x1 + ts[0] + 4, y1), color, -1)
        cv2.putText(frame, text, (x1 + 2, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

        # Fill bar
        bar_x = x1 - 12
        bar_h = y2 - y1
        fill_h = int(bar_h * pct / 100)
        cv2.rectangle(frame, (bar_x, y1), (bar_x + 8, y2), (100, 100, 100), 1)
        if fill_h > 0:
            cv2.rectangle(frame, (bar_x, y2 - fill_h), (bar_x + 8, y2), COLORS["Liquid"], -1)

    # Unmatched liquid boxes
    matched_ids = set(id(liquid_matches[k]) for k in liquid_matches)
    for lb in liquid_boxes:
        if id(lb) not in matched_ids:
            lx1, ly1, lx2, ly2 = map(int, lb)
            cv2.rectangle(frame, (lx1, ly1), (lx2, ly2), COLORS["Liquid"], 2)
            cv2.putText(frame, "Liquid", (lx1, ly1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLORS["Liquid"], 1)

    return frame, tip_count, liquid_count


# --- Process video ---
cap = cv2.VideoCapture(VIDEO_PATH)
fw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
fh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
src_fps = cap.get(cv2.CAP_PROP_FPS) or 30
print(f"Video: {fw}x{fh}, {total_frames} frames, {src_fps:.1f} fps")

writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), src_fps, (fw, fh))
fps_list = []
frame_num = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_num += 1
    t0 = time.time()

    preds = model.predict(frame, iou=IOU_THRESHOLD, conf=CONF_THRESHOLD, fuse_model=False)
    annotated, tip_count, liquid_count = draw_detections(frame, preds, CONF_THRESHOLD)

    dt = time.time() - t0
    fps = 1.0 / dt if dt > 0 else 0
    fps_list.append(fps)
    avg_fps = sum(fps_list[-30:]) / len(fps_list[-30:])

    info = f"FPS: {avg_fps:.1f} | Tips: {tip_count} | Liquid: {liquid_count}"
    cv2.putText(annotated, info, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    writer.write(annotated)

    if frame_num % 100 == 0 or frame_num == 1:
        pct = frame_num / total_frames * 100
        print(f"  Frame {frame_num}/{total_frames} ({pct:.0f}%) | FPS: {avg_fps:.1f} | Tips: {tip_count} Liquid: {liquid_count}")

cap.release()
writer.release()

avg = sum(fps_list) / len(fps_list) if fps_list else 0
print(f"\nDone! Average FPS: {avg:.1f}")
print(f"Output saved to: {OUTPUT_PATH}")

## 6. Play Output Video in Colab

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import os

# Compress for playback (original mp4v may not play in browser)
!ffmpeg -y -i {OUTPUT_PATH} -vcodec libx264 -f mp4 output_playable.mp4 -loglevel quiet

video_path = "output_playable.mp4"
video_size = os.path.getsize(video_path) / (1024 * 1024)
print(f"Output video size: {video_size:.1f} MB")

if video_size < 50:
    # Embed directly in notebook
    with open(video_path, "rb") as f:
        data = b64encode(f.read()).decode()
    HTML(f"""
    <video width="960" controls>
        <source src="data:video/mp4;base64,{data}" type="video/mp4">
    </video>
    """)
else:
    print("Video is too large to embed. Download it using the cell below.")

## 7. Download Output Video

In [ ]:
from google.colab import files
files.download("output_playable.mp4")